# Prompt Engineering Portfolio - Week 1
### GenAI Roadmap - Week 1 Mini-Project

Demonstrates **12 prompt engineering techniques** applied across four task types - text classification, summarization, code generation, and data extraction - with outputs compared across **Groq**, **api.airforce**, and a **local Ollama model**.

This notebook is intentionally thin: prompts live in [`prompts/`](./prompts), reusable logic lives in [`src/`](./src), and each run's results are saved to [`outputs/`](./outputs). See [README.md](./README.md) for the full layout.

| # | Technique | Task type |
|---|-----------|-----------|
| 1 | Zero-Shot Prompting | Text classification |
| 2 | Few-Shot Prompting | Text classification |
| 3 | Chain-of-Thought (CoT) | Data extraction / reasoning |
| 4 | System Message & Role Assignment | Summarization |
| 5 | Structured Output (JSON) | Data extraction |
| 6 | ReAct Pattern (reason + act with a tool) | Reasoning / tool use |
| 7 | Prompt Chaining | Summarization |
| 8 | Code Generation Prompting | Code generation |
| 9 | Reflection / Iterative Refinement | Code generation |
| 10 | Role-Playing / Persona Prompting | Text generation |
| 11 | Self-Consistency (sample & vote) | Reasoning |
| 12 | Cross-Model Comparison | All tasks |

## Setup

1. Copy `.env.example` to `.env` in this folder and fill in your keys:
   - `GROQ_API_KEY`
   - `AIRFORCE_API_KEY`
2. Install dependencies: `pip install -r requirements.txt`
3. For the local model, install [Ollama](https://ollama.ai), run `ollama serve`, then pull a model:
   `ollama pull llama3.1`
4. Run this notebook with its working directory set to `week-1/` (the default when opening it directly from this folder in Jupyter or VS Code) so `prompts` and `src` resolve as importable packages.

In [1]:
import json

import pandas as pd

from src.model_clients import compare_models, show
from src.persistence import save_result
from src.techniques import run_prompt_chain, run_react, run_reflection, run_self_consistency

## 1. Zero-Shot Prompting
*Task: Text classification.* Direct instructions, no examples - the baseline for every task.

In [2]:
from prompts.zero_shot import PROMPT

results = compare_models(PROMPT)
save_result("01_zero_shot", {"prompt": PROMPT, "responses": results})
show(results)

--- Groq ---
Mixed

--- Airforce ---
Mixed.

--- Ollama ---
Mixed



## 2. Few-Shot Prompting
*Task: Text classification.* A handful of labeled examples steers the format and category boundaries.

In [3]:
from prompts.few_shot import PROMPT

results = compare_models(PROMPT)
save_result("02_few_shot", {"prompt": PROMPT, "responses": results})
show(results)

--- Groq ---
The category for the ticket "My invoice shows a $15 charge I don't recognize" is: Billing.

--- Airforce ---
Category: Billing

The ticket is related to a specific charge on an invoice, which falls under billing inquiries or issues.

--- Ollama ---
Billing



## 3. Chain-of-Thought (CoT) Prompting
*Task: Data extraction / arithmetic reasoning.* Explicit "think step by step" reasoning before the final answer.

In [4]:
from prompts.chain_of_thought import PROMPT

results = compare_models(PROMPT)
save_result("03_chain_of_thought", {"prompt": PROMPT, "responses": results})
show(results)

--- Groq ---
To calculate the final total, let's break down the order step by step:

1. Calculate the subtotal for the pizzas: 2 large pizzas at $14.50 each = 2 * $14.50 = $29.00
2. Calculate the subtotal for the sodas: 3 sodas at $2.25 each = 3 * $2.25 = $6.75
3. Calculate the food subtotal: $29.00 (pizzas) + $6.75 (sodas) = $35.75
4. Apply the 10% discount to the food subtotal: $35.75 * 0.10 = $3.58 (discount amount)
5. Calculate the discounted food subtotal: $35.75 - $3.58 = $32.17
6. Add the delivery fee to the discounted food subtotal: $32.17 + $5.00 = $37.17

Final Total: $37.17

--- Airforce ---
To calculate the final total, let's break down the order step by step:

1. **Calculate the subtotal for the pizzas:**
   - 2 large pizzas at $14.50 each = 2 * $14.50 = $29.00

2. **Calculate the subtotal for the sodas:**
   - 3 sodas at $2.25 each = 3 * $2.25 = $6.75

3. **Calculate the food subtotal (pizzas + sodas):**
   - Food subtotal = $29.00 + $6.75 = $35.75

4. **Apply the 10% dis

## 4. System Message & Role Assignment
*Task: Summarization.* A system prompt sets persona and constraints that shape every response.

In [5]:
from prompts.system_role import PROMPT, SYSTEM

results = compare_models(PROMPT, system=SYSTEM)
save_result("04_system_role", {"prompt": PROMPT, "system": SYSTEM, "responses": results})
show(results)

--- Groq ---
Here are 3 key points about Retrieval-Augmented Generation (RAG) for a non-technical executive:

* RAG is a technique that enables our language models to answer questions using information they weren't initially trained on, making them more versatile and up-to-date.
* This approach works by searching an external database for relevant information related to the user's question, and then using that information to provide more accurate and informed answers.
* By using RAG, we can reduce errors, keep our models aligned with the latest information, and avoid the costly and time-consuming process of retraining our models whenever the underlying data changes.

--- Airforce ---
Here are 3 bullet points summarizing Retrieval-Augmented Generation (RAG) for a non-technical executive:

* RAG allows a language model to provide accurate answers to questions using up-to-date information that it wasn't trained on, reducing the risk of outdated or incorrect responses.
* The system searches

## 5. Structured Output (JSON)
*Task: Data extraction.* Requesting a strict schema for reliable, parseable output.

In [6]:
from prompts.structured_output import PROMPT

results = compare_models(PROMPT)
parsed = {}
for name, text in results.items():
    print(f"--- {name} ---")
    try:
        parsed[name] = json.loads(text)
        print(json.dumps(parsed[name], indent=2))
    except json.JSONDecodeError:
        print("(not valid JSON, raw output below)")
        print(text)
    print()

save_result("05_structured_output", {"prompt": PROMPT, "responses": results, "parsed": parsed})

--- Groq ---
{
  "sender_name": "Maria Chen",
  "company": "Nordic Retail Group",
  "requested_product": "SmartLock Pro",
  "quantity": 500,
  "deadline": "September 15th"
}

--- Airforce ---
{
  "sender_name": "Maria Chen",
  "company": "Nordic Retail Group",
  "requested_product": "SmartLock Pro",
  "quantity": 500,
  "deadline": "September 15th"
}

--- Ollama ---
(not valid JSON, raw output below)
```
{
    "sender_name": "Maria Chen",
    "company": "Nordic Retail Group",
    "requested_product": "SmartLock Pro",
    "quantity": 500,
    "deadline": "September 15th"
}
```



WindowsPath('C:/Work/Priyansh-GenAI-roadmap/week-1/outputs/05_structured_output.json')

## 6. ReAct Pattern (Reason + Act)
*Task: Reasoning with tool use.* The model alternates Thought → Action → Observation, calling a `calculator` tool that `src.techniques.run_react` executes on its behalf (via `src.safe_eval`, not `eval`) and feeds back as an observation.

In [7]:
from prompts.react import INSTRUCTIONS, QUESTION

transcript = run_react(QUESTION, INSTRUCTIONS)
save_result("06_react", {"question": QUESTION, "transcript": transcript})
print(transcript)

Answer the question using this exact format:

Thought: reason about what to do next
Action: calculator[expression]
Observation: (this will be filled in by the system — do not write it yourself)
... repeat Thought/Action/Observation as needed ...
Final Answer: the answer

Only use the calculator Action when you need to compute something. Stop right after an Action line and wait for the Observation.

Question: A store buys widgets at $3.20 each and sells them at $5.75 each. If they sell 480 widgets, what is the total profit?
Thought: To find the total profit, we need to calculate the revenue from selling the widgets and subtract the cost of purchasing them. The revenue can be calculated by multiplying the number of widgets sold by the selling price, and the cost can be calculated by multiplying the number of widgets by the buying price. Let's start by calculating the revenue.

Action: calculator[480 * 5.75]

Observation: 

Thought: Now, let's calculate the cost of purchasing the widgets.

## 7. Prompt Chaining
*Task: Summarization → transformation.* Break the task into sequential prompts, each building on the previous output.

In [8]:
from prompts.prompt_chaining import SOURCE_TEXT, summary_prompt, tweet_prompt

result = run_prompt_chain(SOURCE_TEXT, summary_prompt, tweet_prompt)
save_result("07_prompt_chaining", {"source_text": SOURCE_TEXT, **result})
print("Summary:", result["summary"])
print("\nTweet:", result["tweet"])

Summary: Acme Robotics has announced the general availability of its new warehouse picking robot, Falcon-2, which can identify and pick over 1,200 SKUs per hour with improved efficiency and reduced power consumption. The Falcon-2 is expected to ship next quarter and has already shown promising results with early customers, who have reported a reduction in picking errors of more than half.

Tweet: "Meet Falcon-2! Acme's new warehouse robot picks 1,200+ SKUs/hour with reduced errors & power use #RoboticsInLogistics #WarehouseAutomation"


## 8. Code Generation Prompting
*Task: Code generation.* Ask for a function with an explicit contract: signature, docstring, and test cases.

In [9]:
from prompts.code_generation import PROMPT

results = compare_models(PROMPT)
save_result("08_code_generation", {"prompt": PROMPT, "responses": results})
show(results)

--- Groq ---
```python
import re

def is_palindrome(s: str) -> bool:
    """
    Checks if a string is a palindrome, ignoring case, spaces, and punctuation.

    Args:
        s (str): The input string.

    Returns:
        bool: True if the string is a palindrome, False otherwise.

    Examples:
    >>> is_palindrome("A man, a plan, a canal: Panama")
    True
    >>> is_palindrome("Not a palindrome")
    False
    >>> is_palindrome("Was it a car or a cat I saw?")
    True
    """
    # Remove spaces and punctuation, and convert to lower case
    cleaned_s = re.sub('[\W_]+', '', s.lower())
    
    # Compare the cleaned string with its reverse
    return cleaned_s == cleaned_s[::-1]
```

--- Airforce ---
### Palindrome Check Function
```python
import re

def is_palindrome(s: str) -> bool:
    """
    Checks if a string is a palindrome, ignoring case, spaces, and punctuation.

    Args:
        s (str): The input string to check.

    Returns:
        bool: True if the string is a pali

## 9. Reflection / Iterative Refinement
*Task: Code generation.* The model critiques its own first draft (reusing the Technique 8 prompt), then produces a corrected version - catching edge cases a single pass tends to miss.

In [10]:
from prompts.reflection import CODE_GEN_PROMPT, critique_prompt

result = run_reflection(CODE_GEN_PROMPT, critique_prompt)
save_result("09_reflection", result)
print("--- First draft ---\n", result["first_pass"])
print("\n--- Critique + revision ---\n", result["critique"])

--- First draft ---
 ```python
import re

def is_palindrome(s: str) -> bool:
    """
    Checks if a string is a palindrome, ignoring case, spaces, and punctuation.

    Args:
        s (str): The string to check.

    Returns:
        bool: True if the string is a palindrome, False otherwise.

    Examples:
    >>> is_palindrome("A man, a plan, a canal: Panama")
    True
    >>> is_palindrome("Not a palindrome")
    False
    >>> is_palindrome("Was it a car or a cat I saw?")
    True
    """
    # Remove spaces and punctuation, and convert to lower case
    s = re.sub(r'\W+', '', s).lower()
    # Compare the string with its reverse
    return s == s[::-1]

if __name__ == "__main__":
    import doctest
    doctest.testmod()
```

In this code:

- We import the `re` module to use regular expressions for removing spaces and punctuation.
- The `is_palindrome` function takes a string `s` as input, removes spaces and punctuation using `re.sub`, and converts it to lower case.
- It then checks

## 10. Role-Playing / Persona Prompting
*Task: Text generation.* Assigning a persona in the prompt itself (rather than the system message) reshapes tone and style.

In [11]:
from prompts.persona import PROMPT

results = compare_models(PROMPT)
save_result("10_persona", {"prompt": PROMPT, "responses": results})
show(results)

--- Groq ---
WOW, FOLKS! Are you tired of being TIED DOWN by pesky cords and mediocre sound quality?! WELL, SAY GOODBYE TO THOSE DAYS WITH THE MOST AMAZING, THE MOST ASTOUNDING, THE MOST UNBELIEVABLE WIRELESS EARBUDS YOU'VE EVER LAID EARS ON!

These babies have got it ALL! You're talking 30 - THAT'S RIGHT, 30! - HOURS OF BATTERY LIFE! You can listen to your favorite tunes, podcasts, or audiobooks ALL DAY LONG, ALL NIGHT LONG, WITHOUT MISSING A BEAT!

AND IF THAT'S NOT ENOUGH, these earbuds have got ACTIVE NOISE CANCELLATION! That's right, folks, you'll be able to tune out the world and TUNE IN to your favorite sounds like never before! Whether you're on a noisy commute, at the gym, or just trying to relax, these earbuds have got you covered!

BUT WAIT, THERE'S MORE! These earbuds come with a COMPACT CHARGING CASE that's so small, you can take it with you ANYWHERE! You'll never have to worry about running out of juice on the go again!

DON'T WAIT ANY LONGER, FOLKS! ORDER NOW and experie

## 11. Self-Consistency
*Task: Reasoning.* Sample the same reasoning prompt multiple times and take the majority answer - more robust than trusting a single completion.

In [12]:
from prompts.self_consistency import PROMPT, QUESTION

result = run_self_consistency(PROMPT)
save_result("11_self_consistency", {"question": QUESTION, **result})
for i, s in enumerate(result["samples"], 1):
    print(f"Sample {i}:\n{s}\n")
print("Answer distribution:", result["distribution"])
print("Majority answer:", result["majority"])

Sample 1:
To find the arrival time, we need to add the travel time to the departure time. 

First, convert the travel time to just hours and minutes: 2 hours 40 minutes.
Next, add this to the departure time: 3:15 pm + 2 hours = 5:15 pm.
Then, add the remaining 40 minutes: 5:15 pm + 40 minutes = 5:55 pm.

Answer: 5:55 PM

Sample 2:
To find the arrival time, first convert the travel time to just hours and minutes. The travel time is 2 hours 40 minutes. 

Next, add this travel time to the initial departure time. The departure time is 3:15pm. 

Adding 2 hours to 3:15pm results in 5:15pm. Then, adding the remaining 40 minutes results in 5:55pm.

Answer: 5:55 PM

Sample 3:
To find the arrival time, we need to add the travel time to the departure time. 
The departure time is 3:15pm. 
The travel time is 2 hours 40 minutes. 

First, add 2 hours to 3:15pm: 3:15pm + 2 hours = 5:15pm. 
Then, add 40 minutes to 5:15pm: 5:15pm + 40 minutes = 5:55pm.

Answer: 5:55 PM

Sample 4:
To find the arrival tim

## 12. Cross-Model Comparison
*Task: All task types.* The same prompt run through Groq, api.airforce, and a local Ollama model side by side, so differences in style, accuracy, and verbosity are easy to compare directly.

In [13]:
from prompts.cross_model import PROMPT

results = compare_models(PROMPT)
save_result("12_cross_model", {"prompt": PROMPT, "responses": results})
df = pd.DataFrame(list(results.items()), columns=["Model", "Response"])
df

,Model,Response
0,Groq,The attention mechanism in a transformer is a ...
1,Airforce,The attention mechanism in a transformer is a ...
2,Ollama,"A transformer's ""attention mechanism"" is like ..."


## Takeaways

_Fill this in after running the notebook with real API keys:_

- Which technique improved output quality the most for each task type?
- Where did Groq, api.airforce, and the local Ollama model disagree or differ in quality?
- Which technique was most worth the added prompt complexity, and which wasn't?